# B · Compare host feature profiles by pathogen group

This notebook compares the **resolved question sets** for every registered host and pathogen group. Each pathogen group gets a square host × host matrix. A cell is the Jaccard similarity of the two hosts’ applicable feature sets:

\[
J(A,B) = |A ∩ B| / |A ∪ B|

A value of 1 means the resolved profiles contain the same features; 0 means they share none. Feature identity includes `feature_id`, `question_version`, and `scope`, so a changed question version is not silently treated as identical. Host and pathogen applicability is resolved with the same `resolve_profile` function used by the V4 workflow.

By default, the matrix includes all resolved scopes. Set `SCOPE_FILTER = {"pathogen_host_pair"}` below to compare only host-pair questions; this avoids pathogen-level traits that are shared across hosts by design from inflating the similarity. A zero-union comparison is left undefined (`NaN`).


In [ ]:
import sys
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pathogen_atlas_v4").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pathogen_atlas_v4").is_dir():
    raise RuntimeError("Open this notebook from the Pathogen Atlas V4 repository")
sys.path.insert(0, str(PROJECT_ROOT))

from pathogen_atlas_v4.features import resolve_profile
from pathogen_atlas_v4.host_taxonomy.ncbi import PILOT_HOSTS

HOSTS_PATH = PROJECT_ROOT / "configs/host_choices.yaml"
PATHOGENS_PATH = PROJECT_ROOT / "configs/pathogen_taxonomies.yaml"
FEATURES_PATH = PROJECT_ROOT / "configs/features.yaml"
with HOSTS_PATH.open(encoding="utf-8") as stream:
    host_registry = yaml.safe_load(stream)
with PATHOGENS_PATH.open(encoding="utf-8") as stream:
    pathogen_registry = yaml.safe_load(stream)
with FEATURES_PATH.open(encoding="utf-8") as stream:
    feature_catalogue = yaml.safe_load(stream)

RELATIONSHIP_CONTEXT = "disease"
PROFILE_VERSION = str(feature_catalogue.get("profile_version", ""))
# None includes every resolved scope; use {"pathogen_host_pair"} for host-specific questions only.
SCOPE_FILTER = None


## 1 · Resolve each host × pathogen-group question set

Host aliases are collapsed by NCBI taxon ID, matching the question-generation notebook. The resolver applies curated host-group traits, host-taxon traits, pathogen-group traits, and the disease relationship context before collecting feature IDs.


In [ ]:
# Deduplicate host aliases while keeping the first configured display name.
host_aliases_by_id = {}
for name, taxon_id in host_registry["host_name_to_id"].items():
    host_aliases_by_id.setdefault(str(taxon_id), []).append(str(name))

hosts = []
for host_id, aliases in host_aliases_by_id.items():
    taxon_id = host_id.removeprefix("ncbi:")
    hosts.append({
        "host_id": host_id,
        "taxon_id": taxon_id,
        "label": aliases[0],
        "aliases": aliases[1:],
        "host_group": PILOT_HOSTS.get(taxon_id, "other"),
    })
pathogen_groups = [
    {"pathogen_group": str(group_id), "label": str(group.get("label", group_id))}
    for group_id, group in pathogen_registry["pathogen_groups"].items()
]

feature_sets = {}
feature_records = {}
profile_rows = []
for group in pathogen_groups:
    group_id = group["pathogen_group"]
    for host in hosts:
        resolved = resolve_profile(
            feature_catalogue,
            pathogen_group=group_id,
            host_group=host["host_group"],
            host_taxon_id=host["taxon_id"],
            relationship_context=RELATIONSHIP_CONTEXT,
            profile_version=PROFILE_VERSION,
        )
        records = resolved["ordered_features"]
        if SCOPE_FILTER is not None:
            records = [feature for feature in records if feature.get("scope") in SCOPE_FILTER]
        # Include version and scope in identity to avoid equating incompatible definitions.
        identities = {
            (str(feature["feature_id"]), str(feature.get("question_version", "")), str(feature["scope"]))
            for feature in records
        }
        feature_sets[(group_id, host["host_id"])] = identities
        feature_records[(group_id, host["host_id"])] = records
        profile_rows.append({
            "pathogen_group": group_id,
            "host": host["label"],
            "host_id": host["host_id"],
            "host_group": host["host_group"],
            "feature_count": len(identities),
            "profile_hash": resolved["resolved_profile_hash"],
        })

profile_counts = pd.DataFrame(profile_rows)
print(f"Resolved {len(pathogen_groups)} pathogen groups × {len(hosts)} distinct hosts.")
display(profile_counts.pivot(index="host", columns="pathogen_group", values="feature_count").reindex(
    index=[host["label"] for host in hosts], columns=[group["pathogen_group"] for group in pathogen_groups]
))


## 2 · Build the Jaccard matrices

Rows and columns use the same configured host order. The intersection and union counts are retained in a pairwise table so each matrix value can be inspected as `shared features / features in either profile`.


In [ ]:
host_labels = [host["label"] for host in hosts]
host_id_by_label = {host["label"]: host["host_id"] for host in hosts}
matrices = {}
pairwise_rows = []

for group in pathogen_groups:
    group_id = group["pathogen_group"]
    matrix = pd.DataFrame(np.nan, index=host_labels, columns=host_labels, dtype=float)
    for left, right in combinations(hosts, 2):
        left_set = feature_sets[(group_id, left["host_id"])]
        right_set = feature_sets[(group_id, right["host_id"])]
        intersection = left_set & right_set
        union = left_set | right_set
        score = len(intersection) / len(union) if union else np.nan
        matrix.loc[left["label"], right["label"]] = score
        matrix.loc[right["label"], left["label"]] = score
        pairwise_rows.append({
            "pathogen_group": group_id,
            "host_a": left["label"],
            "host_b": right["label"],
            "features_a": len(left_set),
            "features_b": len(right_set),
            "intersection": len(intersection),
            "union": len(union),
            "jaccard": score,
            "shared_feature_ids": "; ".join(sorted(item[0] for item in intersection)),
        })
    for host in hosts:
        own_set = feature_sets[(group_id, host["host_id"])]
        matrix.loc[host["label"], host["label"]] = 1.0 if own_set else np.nan
    matrices[group_id] = matrix

pairwise_comparisons = pd.DataFrame(pairwise_rows)


## 3 · Visualize one square matrix per pathogen group

The common 0–1 color scale makes values comparable between panels. Each cell is annotated with its Jaccard fraction; diagonal cells are 1 for nonempty profiles.


In [ ]:
for group in pathogen_groups:
    group_id = group["pathogen_group"]
    matrix = matrices[group_id]
    fig, ax = plt.subplots(figsize=(10, 9))
    image = ax.imshow(matrix.to_numpy(dtype=float), vmin=0, vmax=1, cmap="YlGnBu")
    ax.set_xticks(range(len(host_labels)), host_labels, rotation=45, ha="right")
    ax.set_yticks(range(len(host_labels)), host_labels)
    ax.set_title(f"{group['label']} · host feature-set similarity (Jaccard)", pad=14)
    ax.set_xlabel("Host")
    ax.set_ylabel("Host")
    ax.set_aspect("equal")
    for row in range(len(host_labels)):
        for col in range(len(host_labels)):
            value = matrix.iat[row, col]
            if pd.notna(value):
                ax.text(col, row, f"{value:.2f}", ha="center", va="center",
                        color="white" if value >= 0.58 else "#183247", fontsize=8)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="Jaccard similarity")
    fig.tight_layout()
    plt.show()


## 4 · Inspect pairwise overlaps

Use this table to see the numerator and denominator behind each off-diagonal cell, plus the shared IDs. Sort by pathogen group or Jaccard value to find the most similar and most different host profiles.


In [ ]:
display(
    pairwise_comparisons.sort_values(
        ["pathogen_group", "jaccard", "host_a", "host_b"],
        ascending=[True, False, True, True],
        na_position="last",
    ).reset_index(drop=True)
)


## Interpretation notes

- Jaccard compares **which questions apply**, not their answers or the amount of evidence collected.
- A shared feature means the same `feature_id`, `question_version`, and `scope` resolved for both hosts.
- `pathogen`-scope features may apply identically across hosts; use `SCOPE_FILTER = {"pathogen_host_pair"}` if the question is specifically how host-dependent the profiles are.
- A value of 0 means the two resolved sets have no feature identities in common. An empty union is undefined, not zero.
- The host trait assignments and feature applicability rules are curated inputs. A missing host trait can exclude a question from that host’s set.


## 5 · Compare two hosts in one shared IMSTE space

Choose two distinct hosts and a pathogen group, then run the comparison. The notebook resolves the applicable feature sets for both hosts and keeps only their feature-identity intersection. It uses accepted/supported pathogen–host associations from the built Explorer payload.

Each pathogen gets one vector. If it is associated with both hosts, feature values from its two pair profiles are averaged when both are available; if only one value is available, that value is used. Missing values are never treated as zero. To fit IMSTE without imputation, the notebook further keeps only shared features with observed values for every included pathogen, then drops constant columns. The fit reports the final feature and pathogen counts.

Marker meanings are: **circle** = accepted for both hosts, **square** = accepted for the first host only, **diamond** = accepted for the second host only. “Only” means no accepted/supported association is recorded for the other host; the hover tooltip shows that host’s actual status (which may be rejected or not screened).


In [ ]:
import gzip
import json
import math

import ipywidgets as widgets
import plotly.express as px
from IPython.display import clear_output, display
from sklearn.preprocessing import StandardScaler

from imste import IteratedMinimumSpanningTreeEmbedder

SITE_DIST = PROJECT_ROOT / "sites/pathogen-atlas-explorer/dist"
with gzip.open(SITE_DIST / "explorer-data.json.gz", "rt", encoding="utf-8") as stream:
    explorer_metadata = json.load(stream)

HOST_OPTIONS = [(host["label"], host["host_id"]) for host in hosts]
HOST_DEFAULT_1 = next((host["host_id"] for host in hosts if host["host_id"] == "ncbi:9606"), hosts[0]["host_id"])
HOST_DEFAULT_2 = next((host["host_id"] for host in hosts if host["host_id"] == "ncbi:4530"),
                      next(host["host_id"] for host in hosts if host["host_id"] != HOST_DEFAULT_1))
GROUP_OPTIONS = [(group["label"], group["pathogen_group"]) for group in pathogen_groups]
GROUP_DEFAULT = next((group["pathogen_group"] for group in pathogen_groups
                      if group["pathogen_group"] == "viruses"), pathogen_groups[0]["pathogen_group"])

host_a = widgets.Dropdown(options=HOST_OPTIONS, value=HOST_DEFAULT_1, description="Host 1:", layout=widgets.Layout(width="360px"))
host_b = widgets.Dropdown(options=HOST_OPTIONS, value=HOST_DEFAULT_2, description="Host 2:", layout=widgets.Layout(width="360px"))
pathogen_group = widgets.Dropdown(options=GROUP_OPTIONS, value=GROUP_DEFAULT, description="Group:", layout=widgets.Layout(width="320px"))
run_button = widgets.Button(description="Fit shared-feature IMSTE", button_style="success", icon="area-chart")
comparison_output = widgets.Output()

GROUP_SHARDS = {}
def load_group_rows(group_id):
    if group_id not in GROUP_SHARDS:
        shard_path = SITE_DIST / "taxa" / f"{group_id}.json.gz"
        with gzip.open(shard_path, "rt", encoding="utf-8") as stream:
            packed_rows = json.load(stream)
        columns = explorer_metadata["species_columns"]
        GROUP_SHARDS[group_id] = [dict(zip(columns, values)) for values in packed_rows]
    return GROUP_SHARDS[group_id]

HOST_BY_ID = {host["host_id"]: host for host in hosts}
SUPPORTED_STATUSES = {"accepted", "supported"}
def is_supported(pair):
    return str((pair or {}).get("pair_selection_status", "")).casefold() in SUPPORTED_STATUSES

def resolved_feature_signatures(host_id, group_id):
    host = HOST_BY_ID[host_id]
    resolved = resolve_profile(
        feature_catalogue,
        pathogen_group=group_id,
        host_group=host["host_group"],
        host_taxon_id=host["taxon_id"],
        relationship_context=RELATIONSHIP_CONTEXT,
        profile_version=PROFILE_VERSION,
    )
    records = resolved["ordered_features"]
    if SCOPE_FILTER is not None:
        records = [record for record in records if record.get("scope") in SCOPE_FILTER]
    return {
        (str(record["feature_id"]), str(record.get("question_version", "")), str(record["scope"]))
        for record in records
    }

print("Select two different hosts and a pathogen group, then click Fit shared-feature IMSTE.")
display(widgets.HBox([host_a, host_b]), widgets.HBox([pathogen_group, run_button]), comparison_output)


In [ ]:
def fit_and_plot_shared_host_map(_):
    with comparison_output:
        clear_output(wait=True)
        first_id, second_id, group_id = host_a.value, host_b.value, pathogen_group.value
        if first_id == second_id:
            print("Choose two different hosts.")
            return
        first_host, second_host = HOST_BY_ID[first_id], HOST_BY_ID[second_id]
        label_a, label_b = first_host["label"], second_host["label"]

        signatures_a = resolved_feature_signatures(first_id, group_id)
        signatures_b = resolved_feature_signatures(second_id, group_id)
        shared_signatures = signatures_a & signatures_b
        shared_feature_ids = sorted({signature[0] for signature in shared_signatures})
        if not shared_feature_ids:
            print("These host profiles have no applicable shared features for this pathogen group.")
            return

        pathogen_vectors = []
        excluded_no_shared_values = 0
        for taxon in load_group_rows(group_id):
            profiles = taxon.get("pair_profiles") or {}
            pair_a, pair_b = profiles.get(first_id) or {}, profiles.get(second_id) or {}
            affects_a, affects_b = is_supported(pair_a), is_supported(pair_b)
            if not (affects_a or affects_b):
                continue

            values_a = pair_a.get("feature_values") or {} if affects_a else {}
            values_b = pair_b.get("feature_values") or {} if affects_b else {}
            feature_values = {}
            for feature_id in shared_feature_ids:
                observed = []
                for source in (values_a, values_b):
                    value = source.get(feature_id)
                    try:
                        numeric = float(value)
                    except (TypeError, ValueError):
                        continue
                    if math.isfinite(numeric):
                        observed.append(numeric)
                if observed:
                    feature_values[feature_id] = float(sum(observed) / len(observed))
            if not feature_values:
                excluded_no_shared_values += 1
                continue

            if affects_a and affects_b:
                class_name = "Both hosts"
            elif affects_a:
                class_name = f"First only · {label_a}"
            else:
                class_name = f"Second only · {label_b}"
            pathogen_vectors.append({
                "pathogen_id": str(taxon["organism_id"]),
                "pathogen": str(taxon["accepted_name"]),
                "class": class_name,
                "feature_values": feature_values,
                "host_1_status": pair_a.get("pair_selection_status", "not screened"),
                "host_2_status": pair_b.get("pair_selection_status", "not screened"),
                "host_1_supporting_pmids": pair_a.get("n_jev_accepted_pmids", pair_a.get("n_pathogen_pmids", 0)),
                "host_2_supporting_pmids": pair_b.get("n_jev_accepted_pmids", pair_b.get("n_pathogen_pmids", 0)),
            })

        if len(pathogen_vectors) < 3:
            print(f"Only {len(pathogen_vectors)} supported pathogen(s) have any scored shared-feature values; IMSTE needs at least 3.")
            print(f"Applicable shared features: {len(shared_feature_ids)} | no shared-feature values: {excluded_no_shared_values}")
            return

        # Match the project embedding workflow: retain features observed for every fitted pathogen;
        # do not turn unknown or unscored values into biological zeros.
        complete_feature_ids = [
            feature_id for feature_id in shared_feature_ids
            if all(feature_id in row["feature_values"] for row in pathogen_vectors)
        ]
        if not complete_feature_ids:
            print("No shared features have observed values for every supported pathogen in this selection.")
            print("Try another host pair or pathogen group; missing feature values are not imputed.")
            print(f"Supported pathogens with any shared-feature values: {len(pathogen_vectors)}")
            return

        matrix = np.asarray([
            [row["feature_values"][feature_id] for feature_id in complete_feature_ids]
            for row in pathogen_vectors
        ], dtype=float)
        variable_mask = np.nanstd(matrix, axis=0) > 0
        matrix = matrix[:, variable_mask]
        used_feature_ids = [feature_id for feature_id, keep in zip(complete_feature_ids, variable_mask) if keep]
        if not used_feature_ids:
            print("The complete shared feature values are constant; IMSTE has no variable dimensions to fit.")
            return
        if len(pathogen_vectors) < 3:
            print("IMSTE needs at least 3 complete pathogens.")
            return
        matrix = StandardScaler().fit_transform(matrix)
        n_species = len(pathogen_vectors)
        n_msts = min(10, max(2, n_species - 3))
        mapper = IteratedMinimumSpanningTreeEmbedder(
            n_msts=n_msts, n_components=2, n_epochs=300, batch_size=32768,
            learning_rate=0.05, negative_ratio=5, lambda_rep=0.5,
            random_state=42, epsilon=0.0001, device="cpu", logistic_margin=1.0,
            logistic_temperature=0.5, mst_method="famst",
            mst_neighbors=min(15, n_species - 1), mst_inter_component_edges=5,
            mst_max_neighbors=None, mst_representatives_per_component=10,
        )
        try:
            coordinates = np.asarray(mapper.fit_transform(matrix), dtype=float)
        except (ValueError, RuntimeError, FloatingPointError) as error:
            print(f"IMSTE could not fit this selection: {error}")
            return
        if coordinates.shape != (n_species, 2) or not np.isfinite(coordinates).all():
            print("IMSTE returned invalid coordinates for this selection.")
            return
        if float(np.max(coordinates.std(axis=0))) < 1e-6:
            print("IMSTE returned a collapsed map for this selection.")
            return

        for row, (x, y) in zip(pathogen_vectors, coordinates):
            row["IMSTE 1"], row["IMSTE 2"] = float(x), float(y)
        plot_data = pd.DataFrame(pathogen_vectors)
        symbols = {"Both hosts": "circle", f"First only · {label_a}": "square",
                   f"Second only · {label_b}": "diamond"}
        colors = {"Both hosts": "#16877d", f"First only · {label_a}": "#315f8c",
                  f"Second only · {label_b}": "#d28732"}
        fig = px.scatter(
            plot_data, x="IMSTE 1", y="IMSTE 2", color="class", symbol="class",
            symbol_map=symbols, color_discrete_map=colors,
            hover_name="pathogen",
            hover_data={"pathogen_id": True, "host_1_status": True, "host_2_status": True,
                        "host_1_supporting_pmids": True, "host_2_supporting_pmids": True,
                        "class": True, "IMSTE 1": ':.3f', "IMSTE 2": ':.3f'},
            title=f"{group_id.title()} · shared-feature IMSTE · {label_a} vs {label_b}",
            labels={"class": "Confirmed disease association", "IMSTE 1": "IMSTE 1", "IMSTE 2": "IMSTE 2"},
        )
        fig.update_traces(marker={"size": 11, "line": {"width": 0.7, "color": "white"}})
        fig.update_layout(width=950, height=760, legend_title_text="Confirmed association",
                          xaxis_title="IMSTE 1", yaxis_title="IMSTE 2",
                          margin={"l": 55, "r": 30, "t": 80, "b": 55})
        display(fig)
        print(f"Pathogen group: {group_id} | hosts: {label_a} × {label_b}")
        print(f"Applicable feature intersection: {len(shared_feature_ids)}")
        print(f"Complete, variable features used: {len(used_feature_ids)} — {', '.join(used_feature_ids)}")
        print(f"Pathogens plotted: {len(plot_data)} | pathogens with no shared-feature values: {excluded_no_shared_values} | n_msts: {n_msts}")

run_button.on_click(fit_and_plot_shared_host_map)
